# ML Assignment 2


In [1]:
!pip install scipy==1.11.4 scikit-learn==1.4.2

In [3]:
import pandas as pd
import numpy as np
import os, joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef

# ==================================================
# DATA LOADING AND PREPROCESSING
# ==================================================
#
# Load dataset
df=pd.read_csv('dataset.csv')
#
# Handle missing BMI values
if 'bmi' in df.columns: df['bmi']=df['bmi'].fillna(df['bmi'].median())#
# Encode categorical features into numeric values
for col in df.columns:
    if df[col].dtype=='object':
        enc=LabelEncoder()
        df[col]=enc.fit_transform(df[col].astype(str))
# Check dataset shape
print("="*60)
print("DATA LOADING AND PREPROCESSING")
print("="*60)
print(df.shape)
# Preview processed data
print("="*60)
print("PREVIEW PROCESSED DATA")
print("="*60)
df.head()

DATA LOADING AND PREPROCESSING
(5110, 12)
PREVIEW PROCESSED DATA


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,1,67.0,0,1,1,2,1,228.69,36.6,1,1
1,51676,0,61.0,0,0,1,3,0,202.21,28.1,2,1
2,31112,1,80.0,0,1,1,2,0,105.92,32.5,2,1
3,60182,0,49.0,0,0,1,2,1,171.23,34.4,3,1
4,1665,0,79.0,1,0,1,3,0,174.12,24.0,2,1


In [5]:
# Import LabelEncoder for converting categorical text data into numeric values
from sklearn.preprocessing import LabelEncoder

# List of categorical columns that need to be encoded
categorical_columns = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status"
]

# Apply Label Encoding to each categorical feature
for col in categorical_columns:
    le = LabelEncoder()
    # Convert categorical values (text) into numerical labels
    # Example: Male -> 1, Female -> 0
    df[col] = le.fit_transform(df[col].astype(str))

print("="*60)
print("DIPLAY DF TYPES AFTER PROCESSING")
print("="*60)
print(df.dtypes)

# ==================================================
# FEATURE AND TARGET SEPARATION
# ==================================================

# Define the target column to be predicted
TARGET = "stroke"

# Create feature matrix (X) by removing the target column
# X contains all input features used for prediction
X = df.drop(TARGET, axis=1)

# Create target variable (y)
# y contains the stroke labels (0 = No Stroke, 1 = Stroke)
y = df[TARGET]

# Import function for splitting dataset into training and testing sets
from sklearn.model_selection import train_test_split


# Split the dataset into training and testing data
# test_size=0.2 -> 20% data for testing, 80% for training
# random_state=42 -> ensures reproducible results
# stratify=y -> preserves the class distribution of the target variable
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Display the shape of training and testing datasets
print("="*60)
print("Display the shape of training and testing datasets")
print("="*60)
print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)


print("="*60)
print("Define a collection of classification models that")
print(" will be trained and evaluated for stroke prediction.")
print("="*60)
models={
# Logistic Regression with feature scaling
# class_weight='balanced' helps handle class imbalance
'Logistic Regression':Pipeline([('scaler',StandardScaler()),('clf',LogisticRegression(max_iter=3000,class_weight='balanced'))]),

# Decision Tree Classifier
# random_state ensures reproducible results
# class_weight='balanced' gives more importance to minority class
'Decision Tree':DecisionTreeClassifier(random_state=42,class_weight='balanced'),

# K-Nearest Neighbors (KNN)
# Feature scaling is important because KNN uses distance calculations
'KNN':Pipeline([('scaler',StandardScaler()),('clf',KNeighborsClassifier())]),

# Gaussian Naive Bayes Classifier
# Feature scaling is applied before model training
'Naive Bayes':Pipeline([('scaler',StandardScaler()),('clf',GaussianNB())]),

# Support Vector Machine (SVM)
# probability=True enables probability predictions
# class_weight='balanced' helps address class imbalance
'SVM':Pipeline([('scaler',StandardScaler()),('clf',SVC(probability=True,class_weight='balanced'))])
}

# Display the names of all defined machine learning models
models.keys()

# ==================================================
# MODEL TRAINING, EVALUATION, AND SAVING
# ==================================================

print("="*60)
print("MODEL TRAINING AND EVALUATION")
print("="*60)
print("Five machine learning models will be trained and compared.")

# Create a directory to store trained model files
# exist_ok=True prevents errors if the folder already exists
os.makedirs('model', exist_ok=True)

# Initialize an empty list to store evaluation results
results = []

# Train and evaluate each machine learning model
for name, model in models.items():

    # Display the model currently being trained
    print(f'Training {name}')

    # Train the model using the training dataset
    model.fit(X_train, y_train)

    # Generate predictions on the test dataset
    pred = model.predict(X_test)

    # Calculate ROC-AUC score if probability estimates are available
    try:
        auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    except:
        # Assign 0 if the model does not support probability prediction
        auc = 0

    # Store performance metrics for the current model
    results.append([
        name,
        accuracy_score(y_test, pred),  # Accuracy
        auc,                            # ROC-AUC Score
        precision_score(y_test, pred, average='weighted', zero_division=0),  # Precision
        recall_score(y_test, pred, average='weighted', zero_division=0),     # Recall
        f1_score(y_test, pred, average='weighted', zero_division=0),         # F1 Score
        matthews_corrcoef(y_test, pred)                                       # MCC
    ])

    # Save the trained model as a .pkl file
    # Example: logistic_regression.pkl
    joblib.dump(
        model,
        'model/' + name.lower().replace(' ', '_') + '.pkl'
    )

# Create a DataFrame containing model evaluation results
metric_df = pd.DataFrame(
    results,
    columns=['Model', 'Accuracy', 'AUC', 'Precision', 'Recall', 'F1', 'MCC']
)

# Round metric values to 4 decimal places
metric_df = metric_df.round(4)

# Save evaluation results to a CSV file
metric_df.to_csv('model_metrics.csv', index=False)

# Display model comparison results

print("="*60)
print("DISPLAY MODEL COMPARE RESULTS")
print("="*60)
metric_df
print(metric_df)

print("="*60)
print("Model trained records and test records")
print("="*60)

for name, model in models.items():
    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    print("\n", name)
    print(pd.Series(pred).value_counts())


print("\nINTERPRETATION OF RESULTS")

print("""
1. Logistic Regression achieved the highest AUC score,
   indicating strong capability in distinguishing stroke cases.

2. Decision Tree achieved high accuracy but predicted
   fewer stroke cases.

3. KNN achieved the highest accuracy score but detected
   very few stroke cases, which may indicate bias toward
   the majority class.

4. Naive Bayes provided a balanced performance with
   reasonable AUC and F1 scores.

5. SVM identified more stroke cases and can be useful
   when detecting positive cases is important.

Conclusion:
Accuracy alone is not sufficient for medical datasets.
AUC, Recall and MCC should also be considered while
selecting the best model.
""")


DIPLAY DF TYPES AFTER PROCESSING
id                     int64
gender                 int64
age                  float64
hypertension           int64
heart_disease          int64
ever_married           int64
work_type              int64
Residence_type         int64
avg_glucose_level    float64
bmi                  float64
smoking_status         int64
stroke                 int64
dtype: object
Display the shape of training and testing datasets
Training Data Shape: (4088, 11)
Testing Data Shape: (1022, 11)
Define a collection of classification models that
 will be trained and evaluated for stroke prediction.
MODEL TRAINING AND EVALUATION
Five machine learning models will be trained and compared.
Training Logistic Regression
Training Decision Tree
Training KNN
Training Naive Bayes
Training SVM
DISPLAY MODEL COMPARE RESULTS
                 Model  Accuracy     AUC  Precision  Recall      F1     MCC
0  Logistic Regression    0.7456  0.8364     0.9448  0.7456  0.8175  0.2597
1        Decision

In [8]:
test_data = X_test.copy()
test_data["stroke"] = y_test

test_data.to_csv(
    "test_data.csv",
    index=False
)

print("test_data.csv created")

test_data.csv created
